In [ ]:
import os
from collections import Counter

import numpy as np


In [ ]:
class Document:
    def __init__(self, text, name=""):
        self.name = name
        self.text = text
        self.words = text.split()
        self.formatted_words = []

        for word in self.words:
            word = word.lower()
            word = word.strip()
            word = word.strip(".,!?;:\"'()[]{}")
            if word:
                self.formatted_words.append(word)

        self.vocabulary = set(self.formatted_words) if self.formatted_words else set()
        self.formatted_word_length = len(self.formatted_words)

    def word_frequency(self):
        return Counter(self.formatted_words)


In [ ]:
class Corpus:
    def __init__(self, data_path):
        self.data_path = data_path
        self.documents = []
        self.load_directory()
        self.document_count = len(self.documents)

    def add_document(self, document):
        self.documents.append(document)

    def load_directory(self):
        for filename in sorted(os.listdir(self.data_path)):
            if filename.endswith(".txt"):
                file_path = os.path.join(self.data_path, filename)
                with open(file_path, "r", encoding="utf-8") as file:
                    text = file.read()
                self.add_document(Document(text, filename))

    def vocabulary(self):
        vocab = set()
        for document in self.documents:
            vocab.update(document.vocabulary)
        return vocab

    def total_words(self):
        return sum(document.formatted_word_length for document in self.documents)

    def word_frequencies(self):
        frequencies = Counter()
        for document in self.documents:
            for word, count in document.word_frequency().items():
                frequencies[word] += count
        return frequencies

    def token_to_id(self):
        token_to_id = {}
        for document in self.documents:
            for word in document.formatted_words:
                if word not in token_to_id:
                    token_to_id[word] = len(token_to_id)
        return token_to_id

    def id_to_token(self):
        return {idx: token for token, idx in self.token_to_id().items()}


In [ ]:
corpus = Corpus(data_path="/Users/sujan/Desktop/MTech in AI/NLP/NLP/Chapter3/data")
print("Documents loaded:", corpus.document_count)
print("Vocabulary:", sorted(corpus.vocabulary()))


In [ ]:
word_to_id = corpus.token_to_id()
idx_to_word = {idx: word for word, idx in word_to_id.items()}
print("Word to id:", word_to_id)


In [ ]:
def build_cbow_dataset(corpus, window_size=2):
    dataset = []
    for document in corpus.documents:
        words = document.formatted_words
        for i, target_word in enumerate(words):
            context_words = []
            for offset in range(1, window_size + 1):
                if i - offset >= 0:
                    context_words.append(words[i - offset])
                if i + offset < len(words):
                    context_words.append(words[i + offset])
            if context_words:
                dataset.append((context_words, target_word))
    return dataset

EMBEDDING_SIZE = 10
rng = np.random.default_rng(42)
cbow_dataset = build_cbow_dataset(corpus, window_size=2)
vocab_size = len(word_to_id)
embedding_matrix = rng.normal(0, 0.1, size=(vocab_size, EMBEDDING_SIZE))
output_matrix = rng.normal(0, 0.1, size=(vocab_size, EMBEDDING_SIZE))
learning_rate = 0.05
epochs = 200

for epoch in range(epochs):
    total_loss = 0.0
    for context_words, target_word in cbow_dataset:
        context_ids = [word_to_id[w] for w in context_words]
        target_id = word_to_id[target_word]

        context_vector = embedding_matrix[context_ids].mean(axis=0)
        logits = context_vector @ output_matrix.T
        logits = logits - logits.max()
        probs = np.exp(logits)
        probs = probs / probs.sum()

        loss = -np.log(probs[target_id] + 1e-12)
        total_loss += loss

        grad_logits = probs.copy()
        grad_logits[target_id] -= 1.0

        grad_output = np.outer(grad_logits, context_vector)
        grad_context = (grad_logits @ output_matrix) / len(context_ids)

        output_matrix -= learning_rate * grad_output
        for context_id in context_ids:
            embedding_matrix[context_id] -= learning_rate * grad_context

    if epoch % 50 == 0:
        print(f"Epoch {epoch}: loss = {total_loss:.4f}")

print("Training finished.")
print("Embedding matrix shape:", embedding_matrix.shape)
print("Sample embedding for like:", embedding_matrix[word_to_id['like']])
